In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
!ls /kaggle/input

In [ ]:
import pandas as pd

train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test  = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
sub   = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
print(train.info())

In [ ]:
print(train.isnull().sum())

In [ ]:
print(train['target'].value_counts())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
plot_df = pd.DataFrame({
    'Category': ['Non-Disaster', 'Disaster'],
    'Count': train['target'].value_counts().sort_index().values
})

In [ ]:
plot_df = pd.DataFrame({
    'Category': ['Disaster', 'Non-Disaster'],
    'Count': [
        train['target'].value_counts().sort_index().values[1],  # Disaster = 1
        train['target'].value_counts().sort_index().values[0],  # Non-Disaster = 0
    ]
})

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.barplot(data=plot_df, x='Category', y='Count', palette=['orange', 'teal'])

y_max = plot_df['Count'].max() * 1.1
ax.set_ylim(0, y_max)
for i, v in enumerate(plot_df['Count']):
    ax.text(i, v + 50, str(v), color='black', ha='center', fontweight='bold')

plt.title('Distribution of Tweet Categories', fontsize=14)
plt.ylabel('Number of Tweets')
plt.xlabel('Category')
plt.tight_layout()
plt.show()

In [ ]:
missing_ratio = train.isnull().mean().sort_values(ascending=False)

plt.figure(figsize=(6, 4))
missing_ratio[missing_ratio > 0].plot(kind='bar', color='salmon')
plt.title('Missing Value Ratio per Column', fontsize=14)
plt.ylabel('Missing Ratio')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
train['text_length'] = train['text'].apply(len)
disaster_len = train[train['target'] == 1]['text_length']
non_disaster_len = train[train['target'] == 0]['text_length']

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
bins = range(0, 161, 5)  # ヒストグラムの区切り幅（0〜160）

ax1.hist(disaster_len, bins=bins, color='orange')
ax1.set_title('Disaster Tweets')
ax1.set_xlabel('Tweet Length (chars)')
ax1.set_ylabel('Tweet Count')
ax1.set_xlim(0, 160)

ax2.hist(non_disaster_len, bins=bins, color='teal')
ax2.set_title('Non-Disaster Tweets')
ax2.set_xlabel('Tweet Length (chars)')
ax2.set_ylabel('Tweet Count')
ax2.set_xlim(0, 160)
mean1 = disaster_len.mean()
mean2 = non_disaster_len.mean()

ax1.axvline(mean1, color='black', linestyle='dashed', linewidth=1, label=f'Mean: {mean1:.1f}')
ax2.axvline(mean2, color='black', linestyle='dashed', linewidth=1, label=f'Mean: {mean2:.1f}')

ax1.legend(loc='upper left')
ax2.legend(loc='upper left')

plt.suptitle('Distribution of Tweet Lengths by Class', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
train['word_count'] = train['text'].apply(lambda x: len(str(x).split()))

disaster_wc = train[train['target']==1]['word_count']
non_disaster_wc = train[train['target']==0]['word_count']

In [ ]:
import matplotlib.pyplot as plt

disaster_wc = train[train['target'] == 1]['word_count']
non_disaster_wc = train[train['target'] == 0]['word_count']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
bins = range(0, 36, 1)

# === Disaster Tweets ===
ax1.hist(disaster_wc, bins=bins, color='orange')
ax1.set_title('Disaster Tweets')
ax1.set_xlabel('Word Count')
ax1.set_ylabel('Tweet Count')
ax1.set_xlim(0, 35)

mean1 = disaster_wc.mean()
ax1.axvline(mean1, color='black', linestyle='dashed', linewidth=1, label=f'Mean: {mean1:.1f}')
ax1.legend(loc='upper right')

# === Non-Disaster Tweets ===
ax2.hist(non_disaster_wc, bins=bins, color='teal')
ax2.set_title('Non-Disaster Tweets')
ax2.set_xlabel('Word Count')
ax2.set_ylabel('Tweet Count')
ax2.set_xlim(0, 35)

mean2 = non_disaster_wc.mean()
ax2.axvline(mean2, color='black', linestyle='dashed', linewidth=1, label=f'Mean: {mean2:.1f}')
ax2.legend(loc='upper right')

fig.suptitle('Distribution of Word Count in Tweets', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
keyword_counts = train['keyword'].dropna().value_counts().head(20)

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(x=keyword_counts.values, y=keyword_counts.index, palette='viridis')

plt.title('Top 20 Most Frequent Disaster Keywords (Non-Null)', fontsize=14)

plt.xlabel('Frequency')
plt.ylabel('Keyword')

for i, (keyword, count) in enumerate(keyword_counts.items()):
    plt.text(count + 0.5, i, str(count), va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
top_keywords = keyword_counts.index.tolist()

disaster_counts = train[train['target'] == 1]['keyword'].value_counts()
non_disaster_counts = train[train['target'] == 0]['keyword'].value_counts()

disaster_data = [disaster_counts.get(k, 0) for k in top_keywords]
non_disaster_data = [non_disaster_counts.get(k, 0) for k in top_keywords]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)

sns.barplot(x=disaster_data, y=top_keywords, palette='viridis', ax=axes[0])
axes[0].set_title('Disaster (target=1)', fontsize=14)
axes[0].set_xlabel('Frequency')
axes[0].set_ylabel('Keyword')
for i, count in enumerate(disaster_data):
    axes[0].text(count + 0.5, i, str(count), va='center', fontsize=10)

sns.barplot(x=non_disaster_data, y=top_keywords, palette='rocket', ax=axes[1])
axes[1].set_title('Non-Disaster (target=0)', fontsize=14)
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('')  
for i, count in enumerate(non_disaster_data):
    axes[1].text(count + 0.5, i, str(count), va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
word_1 = train[train['target']==1]['text'].str.split().apply(lambda x : [len(i) for i in x])
word_1

In [ ]:
word_0 = train[train['target']==0]['text'].str.split().apply(lambda x : [len(i) for i in x])
word_0

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


word_1 = train[train['target'] == 1]['text'].str.split().apply(lambda x: [len(i) for i in x])
word_0 = train[train['target'] == 0]['text'].str.split().apply(lambda x: [len(i) for i in x])


avg_1 = word_1.map(np.mean)
avg_0 = word_0.map(np.mean)

mean_1 = avg_1.mean()
mean_0 = avg_0.mean()


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))


sns.histplot(avg_1, ax=ax1, kde=True, color='orange', bins=30)
ax1.axvline(mean_1, color='black', linestyle='--', label=f'Mean: {mean_1:.1f}')
ax1.set_title('Disaster Tweets')
ax1.set_xlabel('Average Word Length')
ax1.set_ylabel('Density')
ax1.legend()


sns.histplot(avg_0, ax=ax2, kde=True, color='teal', bins=30)
ax2.axvline(mean_0, color='black', linestyle='--', label=f'Mean: {mean_0:.1f}')
ax2.set_title('Non-Disaster Tweets')
ax2.set_xlabel('Average Word Length')
ax2.set_ylabel('Density')
ax2.legend()


fig.suptitle('Average Word Length in Each Tweet (by Class)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import nltk
from nltk.corpus import stopwords
from collections import defaultdict

nltk.download('stopwords', quiet=True)  
stop = set(stopwords.words('english'))

def count_stopwords(corpus):
    dic = defaultdict(int)
    for word in corpus:
        if word in stop:
            dic[word] += 1
    return dic

In [ ]:
def create_corpus(target):
    corpus = []
    for x in train[train['target'] == target]['text'].str.split():
        for i in x:
            corpus.append(i)
    return corpus

In [ ]:
corpus = create_corpus(0)
count_result = count_stopwords(corpus)
count_result

In [ ]:
top = sorted(count_result.items(), key=lambda x: x[1], reverse=True)[:20]

In [ ]:
print("Top 20 Stopwords:\n")
for word, freq in top:
    print(f"{word:<10} {freq}")

In [ ]:
import matplotlib.pyplot as plt

x, y = zip(*top)

plt.figure(figsize=(14, 6)) 
plt.bar(x, y, color='teal')  
plt.title("Top 20 Stopwords in target=0", fontsize=16)

plt.xticks(rotation=0, ha='center', fontsize=15)  
plt.tight_layout()  
plt.show()

In [ ]:
corpus_1 = create_corpus(1)
count_result_1 = count_stopwords(corpus_1)
count_result

In [ ]:
top_1 = sorted(count_result_1.items(), key=lambda x: x[1], reverse=True)[:20]

print("Top 20 Stopwords in target=1:\n")
for word, freq in top_1:
    print(f"{word:<10} {freq}")

In [ ]:
import matplotlib.pyplot as plt

top_1 = sorted(count_result_1.items(), key=lambda x: x[1], reverse=True)[:20]

x, y = zip(*top_1)

plt.figure(figsize=(14, 6))
plt.bar(x, y, color='orange') 

plt.xticks(rotation=0, ha='center', fontsize=15)
plt.title("Top 20 Stopwords in target=1", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
corpus_1 = create_corpus(1)

from collections import defaultdict
import string

dic_1 = defaultdict(int)
special = string.punctuation

for i in corpus_1:
    if i in special:
        dic_1[i] += 1

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
x, y = zip(*dic_1.items())
plt.bar(x, y, color='orange')
plt.title("Punctuation Frequency in target=1", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
corpus_0 = create_corpus(0)  

from collections import defaultdict
import string

dic_0 = defaultdict(int)
special = string.punctuation

for i in corpus_0:
    if i in special:
        dic_0[i] += 1

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
x, y = zip(*dic_0.items())
plt.bar(x, y, color='teal')
plt.title("Punctuation Frequency in target=0", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
def get_top_bigrams(corpus, n=20):

    vec = CountVectorizer(ngram_range=(2, 2)).fit(corpus)
    bag_of_words = vec.transform(corpus)

    sum_words = bag_of_words.sum(axis=0)

    words_freq = [
        (word, sum_words[0, idx]) 
        for word, idx in vec.vocabulary_.items()
    ]
    
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    
    return words_freq[:n]

In [ ]:
corpus_1 = train[train['target'] == 1]['text']
top_bigrams_1 = get_top_bigrams(corpus_1, n=20)

for phrase, freq in top_bigrams_1:
    print(f"{phrase}: {freq}")

In [ ]:
corpus_0 = train[train['target'] == 0]['text']
top_bigrams_0 = get_top_bigrams(corpus_0, n=20)

for phrase, freq in top_bigrams_0:
    print(f"{phrase}: {freq}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

phrases_1, freqs_1 = zip(*top_bigrams_1)

plt.figure(figsize=(12, 5))
sns.barplot(x=list(phrases_1), y=list(freqs_1), palette=["orange"]*len(phrases_1))
plt.xticks(rotation=45)
plt.title("Top 20 Bigrams (target = 1)", fontsize=14)
plt.xlabel("Bigram")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

phrases_0, freqs_0 = zip(*top_bigrams_0)

plt.figure(figsize=(12, 5))
sns.barplot(x=list(phrases_0), y=list(freqs_0), palette=["teal"]*len(phrases_0))
plt.xticks(rotation=45)
plt.title("Top 20 Bigrams (target = 0)", fontsize=14)
plt.xlabel("Bigram")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:

!pip uninstall -y transformers


!pip install transformers accelerate --upgrade --quiet

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW  

from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import get_scheduler

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from tqdm import tqdm

In [ ]:
train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
submission = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')

tokenizer = RobertaTokenizer.from_pretrained("roberta-large")
MAX_LEN = 160

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=MAX_LEN)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

X_train, X_val, y_train, y_val = train_test_split(
    train["text"].tolist(),
    train["target"].values,
    test_size=0.2,
    random_state=42,
    stratify=train["target"]
)

train_dataset = TweetDataset(X_train, y_train)
val_dataset = TweetDataset(X_val, y_val)
test_dataset = TweetDataset(test["text"].tolist())

In [ ]:
model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

In [ ]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Train Loss: {avg_loss:.4f}")

 
    model.eval()
    preds, truths = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits
            pred_labels = torch.argmax(logits, dim=1)
            preds.extend(pred_labels.cpu().numpy())
            truths.extend(batch["labels"].cpu().numpy())

    f1 = f1_score(truths, preds)
    print(f"Epoch {epoch+1} - Validation F1 Score: {f1:.4f}")

In [ ]:
model.eval()
test_loader = DataLoader(test_dataset, batch_size=16)
preds = []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        pred_labels = torch.argmax(logits, dim=1)
        preds.extend(pred_labels.cpu().numpy())

submission["target"] = preds
submission.to_csv("submission.csv", index=False)
print("submission.csv saved!")